> 📄 English: [mcp-tutorial.ipynb](mcp-tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial)

# 使用 MCP 构建 Agent：AI 与外部资源的无缝集成

## 简介

Model Context Protocol（MCP）是一个开放协议，旨在标准化应用程序如何为大语言模型（LLM）提供上下文。你可以把 MCP 想象成 AI 应用的 USB-C 接口——正如 USB-C 提供了一种标准化的方式来将设备连接到各种外设，MCP 提供了一种标准化的方式来将 AI 模型连接到不同的数据源和工具。

本教程将指导你在 AI Agent 应用中实现 MCP，展示它如何通过无缝访问外部资源、工具和数据源来增强 Agent 的能力。

## MCP 为何对 Agent 重要

将 AI 模型与外部资源连接的传统方法，往往需要为每个数据源或工具进行定制集成。这会导致：

- **集成复杂度**：每个新数据源都需要独特的实现
- **可扩展性问题**：添加新工具变得越来越困难
- **维护开销**：对一个集成的更新可能破坏其他集成

MCP 通过提供一个标准化协议来解决这些挑战，该协议实现了：

- **统一访问**：多个数据源和工具使用单一接口
- **即插即用的扩展**：轻松添加新能力
- **有状态通信**：AI 与资源之间的实时、双向通信
- **动态发现**：AI 可以即时发现并使用新工具

以下是一段简洁的段落，重点介绍官方 MCP Server 示例：

## 官方 MCP Server 示例

MCP 社区维护了一系列参考服务器实现，展示了最佳实践并演示了各种集成模式。这些官方示例可在 [MCP Servers](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=github-servers-tree&target=https%3A%2F%2Fgithub.com%2Fmodelcontextprotocol%2Fservers%2Ftree%2Fmain%2Fsrc&text=MCP%20Servers) 获取，为希望创建自己的 MCP 服务器的开发者提供了宝贵的起点。

## 我们将构建什么

在本教程中，我们将实现：

1. **构建并使用你自己的 MCP 服务器**：构建一个带定制工具的 MCP 服务器，并连接到 Claude Desktop
2. **启用定制工具的 Agent**：创建一个可以通过 MCP 使用外部工具的定制 Agent

学完本教程，你将理解 MCP 如何通过为 Agent 提供对更广泛数字生态系统的访问来增强其能力，使其更强大、更贴合上下文、更有用。

让我们从理解 MCP 架构和配置环境开始吧！


📖 **想了解更多关于 Model Context Protocol（MCP）及其应用的背景知识，请查看我们详细的博客文章：** [Model Context Protocol (MCP) explained](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=substack-open&target=https%3A%2F%2Fopen.substack.com%2Fpub%2Fdiamantai%2Fp%2Fmodel-context-protocol-mcp-explained%3Fr%3D336pe4%26utm_campaign%3Dpost%26utm_medium%3Dweb%26showWelcomeOnShare%3Dfalse&text=Model%20Context%20Protocol%20%28MCP%29%20explained)

*DiamantAI 是一份专注于 AI 技术、突破和教程的 AI 通讯，订阅人数超过 25,000，位居前 0.1%。*

---


## MCP 架构概览

![MCP Architecture](assets/mcp_architecture.png)

MCP 遵循客户端-服务器架构，包含三个主要组件：

- **Host（宿主）**：需要访问外部资源的 AI 应用（如 Claude Desktop、Cursor 或定制 Agent）
- **Clients（客户端）**：维护与服务器之间连接的连接器
- **Servers（服务器）**：通过 MCP 协议暴露能力（数据、工具、prompts）的轻量程序
- **Data Sources（数据源）**：MCP 服务器可以访问的本地（文件、数据库）和远程服务（API）

MCP 内部的通信使用基于 WebSocket 连接的 JSON-RPC 2.0，确保组件之间的实时双向通信。

## 体验 MCP：先试后建

虽然本教程的重点是构建你自己的 MCP 服务器并将其与 AI Agent 集成，但你可能想在深入开发之前，快速体验一下 MCP 在实际中是如何工作的。

官方 MCP 文档为想要在 Claude Desktop 或其他兼容 AI 应用中试用现有 MCP 服务器的用户，提供了一份优秀的快速入门指南。这让你无需编写任何代码，就能亲身感受 MCP 所带来的能力。

**👉 亲自试试：** [MCP Quick Start Guide for Users](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=modelcontextprotocol-user&target=https%3A%2F%2Fmodelcontextprotocol.io%2Fquickstart%2Fuser&text=MCP%20Quick%20Start%20Guide%20for%20Users)

通过探索快速入门指南，你将对本教程要构建的内容获得实际认识。当你准备好理解其内部原理并创建自己的实现时，请继续下面的一步步开发过程。

现在，让我们开始构建我们自己的 MCP 服务器和客户端！

## 构建你的 MCP 服务器

既然我们已经理解了 MCP 的基础，让我们构建第一个 MCP 服务器！在本节中，我们将使用 CoinGecko API 创建一个加密货币价格查询服务。我们的服务器将提供工具，让 AI 能够查询加密货币的当前价格或市场数据。

### 配置环境

在深入实现之前，让我们安装必要的包并配置环境。

> **注意：** 安装步骤请打开一个终端窗口。这些命令应在常规终端中运行，而不是在 Jupyter notebook 单元格中。

#### 步骤 1：安装 uv 包管理器

```bash
# Run this in your terminal, not in Jupyter
curl -LsSf https://astral.sh/uv/install.sh | sh
```

#### 步骤 2：配置项目

```bash
# Create and navigate to a project directory
mkdir mcp-crypto-server
cd mcp-crypto-server
uv init

# Create and activate virtual environment
uv venv
source .venv/bin/activate  # On Windows: .venv\Scripts\activate

# Install dependencies
# Note: MCP SDK 2.x renamed FastMCP to MCPServer. scripts/mcp_server.py
# uses the 2.x API, so pin >=2.0 rather than an unversioned install.
uv add "mcp[cli]>=2.0" httpx
```

### 运行 MCP 服务器

环境配置好后，我们可以开始构建我们的工具。

请查看 [mcp_server.py](scripts/mcp_server.py) 了解如何构建工具。


现在我们可以通过在终端中运行以下命令来启动服务器：
```bash
# Copy the server file from the scripts folder
cp ../scripts/mcp_server.py .

# Start the MCP server
uv run mcp_server.py
```

### 与 Claude Desktop 集成
如果你还没有下载 Claude Desktop，请查看[此页面](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=claude-download&target=https%3A%2F%2Fclaude.ai%2Fdownload&text=this%20page)。

要将你的 MCP 服务器连接到 Claude Desktop：

#### 步骤 1：找到你的 uv 命令的绝对路径：
```bash
which uv
```
复制输出（例如 /user/local/bin/uv 或类似路径）

#### 步骤 2：创建或编辑 Claude Desktop 配置文件：

- 在 macOS 上：~/Library/Application Support/Claude/claude_desktop_config.json
- 在 Windows 上：%APPDATA%\Claude\claude_desktop_config.json
- 在 Linux 上：~/.config/Claude/claude_desktop_config.json

你可以查看[此页面](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=modelcontextprotocol-user&target=https%3A%2F%2Fmodelcontextprotocol.io%2Fquickstart%2Fuser%232-add-the-filesystem-mcp-server&text=this%20page)了解如何创建配置文件。

#### 步骤 3：添加你的 MCP 服务器配置：

```json
{
    "mcpServers": {
        "crypto-price-tracker": {
            "command": "/ABSOLUTE/PATH/TO/uv",
            "args": [
                "--directory",
                "/ABSOLUTE/PATH/TO/GenAI_Agents/all_agents_tutorials/mcp-crypto-server",
                "run",
                "mcp_server.py"
            ]
        }
    }
}
```
将 `/ABSOLUTE/PATH/TO/uv` 替换为你从 `which uv` 命令得到的路径，将 `/ABSOLUTE/PATH/TO/GenAI_Agents` 替换为你的仓库的绝对路径。


#### 步骤 4：重启 Claude Desktop 以使更改生效。

你应该会在聊天框中看到这个锤子图标。

![Claude Desktop connected with MCP](assets/Claude_Desktop_with_MCP.png)

#### 步骤 5：尝试询问比特币的价格

输入 "What is the current price of Bitcoin ?"，你会得到类似这样的回复：

![Track Bitcoin price with MCP](assets/track_bitcoin_price_with_mcp.png)


恭喜！你已经成功应用了你的 MCP 服务器和工具。现在，你可以尝试向 [mcp_server.py](/mcp-crypto-server/mcp_server.py) 添加你自己的工具。以下是一个示例：

```python
@mcp.tool()
async def get_crypto_market_info(crypto_ids: str, currency: str = "usd") -> str:
    """
    Get market information for one or more cryptocurrencies.

    Parameters:
    - crypto_ids: Comma-separated list of cryptocurrency IDs (e.g., 'bitcoin,ethereum')
    - currency: The currency to display values in (default: 'usd')

    Returns:
    - Market information including price, market cap, volume, and price changes
    """
    # Construct the API URL
    url = f"{COINGECKO_BASE_URL}/coins/markets"

    # Set up the query parameters
    params = {
        "vs_currency": currency,  # Currency to display values in
        "ids": crypto_ids,        # Comma-separated crypto IDs
        "order": "market_cap_desc", # Order by market cap
        "page": 1,                # Page number
        "sparkline": "false"      # Exclude sparkline data
    }

    try:
        # Make the API call
        async with httpx.AsyncClient() as client:
            response = await client.get(url, params=params)
            response.raise_for_status()

            # Parse the response
            data = response.json()

            # Check if we got any data
            if not data:
                return f"No data found for cryptocurrencies: '{crypto_ids}'. Please check the IDs and try again."

            # Format the results
            result = ""
            for crypto in data:
                name = crypto.get('name', 'Unknown')
                symbol = crypto.get('symbol', '???').upper()
                price = crypto.get('current_price', 'Unknown')
                market_cap = crypto.get('market_cap', 'Unknown')
                volume = crypto.get('total_volume', 'Unknown')
                price_change = crypto.get('price_change_percentage_24h', 'Unknown')

                result += f"{name} ({symbol}):\n"
                result += f"Current price: {price} {currency.upper()}\n"
                result += f"Market cap: {market_cap} {currency.upper()}\n"
                result += f"24h trading volume: {volume} {currency.upper()}\n"
                result += f"24h price change: {price_change}%\n\n"

            return result

    except Exception as e:
        return f"Error fetching market data: {str(e)}"
```

用 `uv run mcp_server.py` 重新运行你的 mcp 服务器，重启 Claude Desktop，然后输入 "What's the market data for Dogecoin and Solana?"。你会得到类似这样的回复：

![Track Crypto Market Data with MCP](assets/track_crypto_market_data_with_mcp.png)

## 定制 Agent 通过 MCP 执行工具

构建好我们自己的 MCP 之后，让我们尝试自己构建 MCP Host 和 Client。

### 理解架构

在本节中，我们将构建自己的 MCP Host 和 Client。与此前连接 Claude Desktop 的方式不同，我们现在将创建自己的 Agent，它可以：
1. 充当 MCP Host
2. 从我们的 MCP 服务器发现可用工具
3. 根据用户查询理解何时使用哪个工具
4. 使用合适的参数执行工具
5. 处理工具结果以提供有用的回复

这种架构遵循现代 AI 系统中常见的模式：
- **发现阶段（Discovery Phase）**：我们的定制 host 发现有哪些可用工具
- **规划阶段（Planning Phase）**：Agent 根据用户查询决定使用哪个工具
- **执行阶段（Execution Phase）**：我们的 client 连接到服务器并执行所选工具
- **解释阶段（Interpretation Phase）**：Agent 用自然语言解释结果

以下是一个简单的工作流示意图：

![Track Crypto Market Data with MCP](assets/customized_mcp_host.png)

运行代码前的重要提醒：
⚠️ 别忘了先启动你的 MCP 服务器！⚠️
在运行下面教程中的 Agent 代码之前，请确保你的 MCP 服务器已经启动运行。否则，你的 Agent 将没有任何可发现或执行的工具。

让我们从配置环境和导入必要的库开始：

In [ ]:
! pip install mcp anthropic

我们需要两个主要的库：
- **MCP**：处理与我们 MCP 服务器的客户端-服务器通信，让我们能够构建 host 和 client 组件
- **Anthropic**：与 Claude 交互，为我们的 Agent 推理能力提供动力

现在，让我们为 Agent 配置必要的导入和配置：

In [ ]:
# Import necessary libraries
import os
import json
from typing import List, Dict, Any

# MCP libraries for connecting to server
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Anthropic API for Claude
from anthropic import Anthropic

# Set up Anthropic API key (using the one you provided)
os.environ["ANTHROPIC_API_KEY"] = "your_anthropic_api_key_here"

# Initialize the Anthropic client
client = Anthropic()

# Path to your MCP server
mcp_server_path = "absolute/path/to/your/running/mcp/server"
print("Setup complete!")

我们使用 MCP 的 `stdio_client` 接口，它允许我们连接到作为独立进程运行、并通过标准输入/输出通信的 MCP 服务器。对于本地开发来说，这是一种简单而稳健的方法。通过实现 MCP 协议的双方（host 和 client），我们能够完全掌控 Agent 与 MCP 工具的交互方式。

### 工具发现：构建我们的 MCP Host

构建自定义 MCP 实现的第一步，是创建一个能够从 MCP 服务器发现可用工具的 host。我们的 host 将充当用户、AI 和可用工具之间的中介——类似于 Claude Desktop 的功能，但完全由我们掌控。

让我们实现一个函数，用于连接到我们的 MCP 服务器并发现其工具：

In [ ]:
async def discover_tools():
    """
    Connect to the MCP server and discover available tools.
    Returns information about the available tools.
    """
    # ANSI color codes for better log visibility
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    RESET = "\033[0m"
    SEP = "=" * 40
    
    # Create server parameters for connecting to your MCP server through stdio
    server_params = StdioServerParameters(
        command="python",  # Command to run the server
        args=[mcp_server_path],  # Path to your MCP server script
    )
    
    print(f"{BLUE}{SEP}\n🔍 DISCOVERY PHASE: Connecting to MCP server...{RESET}")
    
    # Connect to the server via stdio
    async with stdio_client(server_params) as (read, write):
        # Create a client session
        async with ClientSession(read, write) as session:
            # Initialize the connection
            print(f"{BLUE}📡 Initializing MCP connection...{RESET}")
            await session.initialize()
            
            # List the available tools
            print(f"{BLUE}🔎 Discovering available tools...{RESET}")
            tools = await session.list_tools()
            
            # Format the tools information for easier viewing
            tool_info = []
            for tool_type, tool_list in tools:
                if tool_type == "tools":
                    for tool in tool_list:
                        tool_info.append({
                            "name": tool.name,
                            "description": tool.description,
                            "schema": tool.inputSchema
                        })
            
            print(f"{GREEN}✅ Successfully discovered {len(tool_info)} tools{RESET}")
            print(f"{SEP}")
            return tool_info

print("Tool discovery function defined")

这个函数充当我们 host 的发现组件：

1. **创建服务器参数**：配置如何启动并连接到 MCP 服务器
2. **建立连接**：使用 `stdio_client` 创建通信通道
3. **初始化会话**：使用通信通道设置 MCP 会话
4. **发现工具**：调用 `list_tools()` 获取所有可用工具
5. **格式化结果**：将工具转换为更易于我们 Agent 使用的格式

我们使用异步方式（`async/await`），因为 MCP 操作在设计上就是非阻塞的。这在 host 实现中很重要，因为它让我们的 Agent 能够并发处理多个操作，即使在等待工具操作完成时也能保持响应。

让我们测试工具发现函数，确保它能正常工作：


In [ ]:
# Test the tool discovery function
tools = await discover_tools()
print(f"Discovered {len(tools)} tools:")
for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool['name']}: {tool['description']}")

运行这段代码时，我们应该能看到 MCP 服务器提供的工具列表。在这里，我们期望看到我们的加密货币工具。

### 工具执行：实现我们的 MCP Client

现在，我们的 host 已经能够发现可用工具，我们需要实现可以执行它们的 client 组件。与可能内置此功能的第三方工具不同，我们自己创建 client 来执行 MCP 工具，从而获得完整的控制和透明度：


In [ ]:
async def execute_tool(tool_name: str, arguments: Dict[str, Any]):
    """
    Execute a specific tool provided by the MCP server.
    
    Args:
        tool_name: The name of the tool to execute
        arguments: A dictionary of arguments to pass to the tool
        
    Returns:
        The result from executing the tool
    """
    # ANSI color codes for better log visibility
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RESET = "\033[0m"
    SEP = "-" * 40
    
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )
    
    print(f"{YELLOW}{SEP}")
    print(f"⚙️ EXECUTION PHASE: Running tool '{tool_name}'")
    print(f"📋 Arguments: {json.dumps(arguments, indent=2)}")
    print(f"{SEP}{RESET}")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # Call the specific tool with the provided arguments
            print(f"{BLUE}📡 Sending request to MCP server...{RESET}")
            result = await session.call_tool(tool_name, arguments)
            
            print(f"{GREEN}✅ Tool execution complete{RESET}")
            
            # Format result preview for cleaner output
            result_preview = str(result)
            if len(result_preview) > 150:
                result_preview = result_preview[:147] + "..."
                
            print(f"{BLUE}📊 Result: {result_preview}{RESET}")
            print(f"{SEP}")
            
            return result

print("Tool execution function defined")

这个函数构成了我们 MCP client 的核心：

1. **连接到服务器**：与我们的发现函数类似，它建立与 MCP 服务器的连接
2. **执行工具**：使用提供的参数调用指定工具
3. **返回结果**：返回工具返回的任何内容

注意，每次执行工具时，我们都会创建一个到 MCP 服务器的新连接。虽然这看起来可能效率不高，但它确保了工具调用之间的清晰隔离，避免了潜在的状态问题。这种无状态方法简化了我们的实现并使其更稳健。在生产系统中，你可能会通过维护持久连接来优化这一点，但当前方法对教学非常有利，因为它清晰地分离了过程中的每一步。

现在，我们有了发现和执行工具的函数，需要将它们与一个能够确定何时以及如何使用这些工具的 AI 集成起来。这就是 Claude 登场的地方。

### 将 AI 与我们的 MCP 实现集成

host 和 client 组件就绪后，我们现在需要将它们与一个能够就工具使用做出智能决策的 AI 系统集成。这是我们自定义 MCP host 的"大脑"，它需要：
1. 根据用户输入理解何时需要工具
2. 为任务选择合适的工具
3. 正确格式化参数
4. 处理并解释结果

让我们实现一个函数来编排这整个过程：

In [ ]:
async def query_claude(prompt: str, tool_info: List[Dict], previous_messages=None):
    """
    Send a query to Claude and process the response.
    
    Args:
        prompt: User's query
        tool_info: Information about available tools
        previous_messages: Previous messages for maintaining context
        
    Returns:
        Claude's response, potentially after executing tools
    """
    # ANSI color codes for better log visibility
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    PURPLE = "\033[95m"
    RESET = "\033[0m"
    SEP = "=" * 40
    
    if previous_messages is None:
        previous_messages = []
    
    print(f"{PURPLE}{SEP}")
    print("🧠 REASONING PHASE: Processing query with Claude")
    print(f"🔤 Query: \"{prompt}\"")
    print(f"{SEP}{RESET}")
    
    # Format tool information for Claude
    tool_descriptions = "\n\n".join([
        f"Tool: {tool['name']}\nDescription: {tool['description']}\nSchema: {json.dumps(tool['schema'], indent=2)}"
        for tool in tool_info
    ])
    
    # Build the system prompt
    system_prompt = f"""You are an AI assistant with access to specialized tools through MCP (Model Context Protocol).
    
Available tools:
{tool_descriptions}

When you need to use a tool, respond with a JSON object in the following format:
{{
    "tool": "tool_name",
    "arguments": {{
        "arg1": "value1",
        "arg2": "value2"
    }}
}}

Do not include any other text when using a tool, just the JSON object.
For regular responses, simply respond normally.
"""
    
    # Filter out system messages from previous messages
    filtered_messages = [msg for msg in previous_messages if msg["role"] != "system"]
    
    # Build the messages for the conversation (WITHOUT system message)
    messages = filtered_messages.copy()
    
    # Add the current user query
    messages.append({"role": "user", "content": prompt})
    
    print(f"{BLUE}📡 Sending request to Claude API...{RESET}")
    
    # Send the request to Claude with system as a top-level parameter
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=4000,
        system=system_prompt,  # System prompt as a separate parameter
        messages=messages      # Only user and assistant messages
    )
    
    # Get Claude's response
    claude_response = response.content[0].text
    print(f"{GREEN}✅ Received response from Claude{RESET}")
    
    # Try to extract and parse JSON from the response
    try:
        # Look for JSON pattern in the response
        import re
        json_match = re.search(r'(\{[\s\S]*\})', claude_response)
        
        if json_match:
            json_str = json_match.group(1)
            print(f"{YELLOW}🔍 Tool usage detected in response{RESET}")
            print(f"{BLUE}📦 Extracted JSON: {json_str}{RESET}")
            
            tool_request = json.loads(json_str)
            
            if "tool" in tool_request and "arguments" in tool_request:
                tool_name = tool_request["tool"]
                arguments = tool_request["arguments"]
                
                print(f"{YELLOW}🔧 Claude wants to use tool: {tool_name}{RESET}")
                
                # Execute the tool using our MCP client
                tool_result = await execute_tool(tool_name, arguments)
                
                # Convert tool result to string if needed
                if not isinstance(tool_result, str):
                    tool_result = str(tool_result)
                
                # Update messages with the tool request and result
                messages.append({"role": "assistant", "content": claude_response})
                messages.append({"role": "user", "content": f"Tool result: {tool_result}"})
                
                print(f"{PURPLE}🔄 Getting Claude's interpretation of the tool result...{RESET}")
                
                # Get Claude's interpretation of the tool result
                final_response = client.messages.create(
                    model="claude-3-5-sonnet-20240620",
                    max_tokens=4000,
                    system=system_prompt,
                    messages=messages
                )
                
                print(f"{GREEN}✅ Final response ready{RESET}")
                print(f"{SEP}")
                
                return final_response.content[0].text, messages
        
    except (json.JSONDecodeError, KeyError, AttributeError) as e:
        print(f"{YELLOW}⚠️ No tool usage detected in response: {str(e)}{RESET}")
    
    print(f"{GREEN}✅ Response ready{RESET}")
    print(f"{SEP}")
    
    return claude_response, messages

print("Claude query function defined")

这个函数通过复杂的推理与执行流程完成了我们自定义 MCP host 的实现：

1. **工具描述**：我们以 Claude 能够理解的方式格式化工具信息
2. **系统提示词**：我们提供关于何时以及如何使用工具的指令
3. **响应分析**：我们在 Claude 的响应中查找 JSON 工具请求
4. **工具执行**：如果检测到工具请求，我们使用 client 执行相应工具
5. **结果处理**：我们将工具结果发回给 Claude 进行解释
6. **对话管理**：我们通过跟踪消息来维护上下文

这创造了一种强大的协同效应：Claude 提供推理和沟通能力，而我们的 MCP 工具提供专业能力和实时数据访问。

让我们用一个关于比特币价格的简单查询来测试我们的 Agent：

In [ ]:
# Run a single query using the tools from your MCP server
query = "What is the current price of Bitcoin?"
print(f"Sending query: {query}")

response, messages = await query_claude(query, tools)
print(f"\nAssistant's response:\n{response}")


运行这个查询时，我们完整的 MCP 实现会遵循以下流程：
1. Claude（通过我们的 host）识别出这是一个关于比特币价格的请求
2. 我们的 AI 决定使用 `get_crypto_price` 工具
3. 它正确格式化参数（使用 "bitcoin" 作为 crypto_id）
4. 我们的 client 连接到服务器并执行工具，返回当前比特币价格
5. Claude 用自然语言结合额外上下文解释结果

这展示了我们 Agent 的全部能力：理解用户意图、选择合适的工具、正确执行它，并提供有用且信息丰富的回复。

### 通过我们的 Client 直接执行工具

虽然我们的集成 MCP host 通常根据用户查询决定使用哪些工具，但有时我们可能希望直接使用 client 执行特定工具。这对于测试我们的 client 实现或演示特定工具功能很有用。让我们创建一个简单示例：

In [ ]:
try:
    # Get the first tool name from your discovered tools
    if tools:
        first_tool = tools[0]
        tool_name = first_tool["name"]
        
        # Use the correct parameter name for get_crypto_price
        arguments = {"crypto_id": "bitcoin"}
        
        print(f"Executing tool '{tool_name}' with arguments: {arguments}")
        result = await execute_tool(tool_name, arguments)
        print(f"Tool result: {result}")
    else:
        print("No tools discovered to test")
except Exception as e:
    print(f"Error executing tool: {str(e)}")

这种直接执行方式对以下场景很有用：
- 单独测试我们的 client 实现
- 调试工具功能
- 构建工具执行预先确定的专业工作流
- 在将 MCP client 与 AI 集成之前，验证它能正常工作

现在，让我们创建一个使用完整 MCP host 实现的交互式聊天界面：

### 构建交互式 MCP Host 界面

对于完整的 MCP host 实现，我们需要一个能够在多轮对话中维护上下文的用户界面。这使我们的 host 能够记住之前的交互，并在后续交流中在此基础上构建，就像 Claude Desktop 等专业 MCP host 一样。让我们实现一个简单的聊天会话函数：

In [ ]:
async def chat_session():
    """
    Run an interactive chat session with the AI agent.
    """
    # ANSI color codes for better log visibility
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    CYAN = "\033[96m"
    BOLD = "\033[1m"
    RESET = "\033[0m"
    SEP = "=" * 50
    
    print(f"{CYAN}{BOLD}{SEP}")
    print("🤖 INITIALIZING MCP AGENT")
    print(f"{SEP}{RESET}")
    
    # Make sure 'tools' is defined from a previous cell, or discover them again
    try:
        # Check if tools is defined and not empty
        if 'tools' not in globals() or not tools:
            print(f"{BLUE}🔍 No tools found, discovering available tools...{RESET}")
            tools_local = await discover_tools()
        else:
            tools_local = tools
            
        print(f"{GREEN}✅ Agent ready with {len(tools_local)} tools:{RESET}")
        
        # Print the available tools for reference
        for i, tool in enumerate(tools_local, 1):
            print(f"{YELLOW}  {i}. {tool['name']}{RESET}")
            print(f"     {tool['description'].strip()}")
        
        # Start the chat session
        print(f"\n{CYAN}{BOLD}{SEP}")
        print(f"💬 INTERACTIVE CHAT SESSION")
        print(f"{SEP}")
        print(f"Type 'exit' or 'quit' to end the session{RESET}")
        
        messages = []
        
        while True:
            # Get user input
            user_input = input(f"\n{BOLD}You:{RESET} ")
            
            # Check if user wants to exit
            if user_input.lower() in ['exit', 'quit']:
                print(f"\n{GREEN}Ending chat session. Goodbye!{RESET}")
                break
            
            # Process the query with Claude
            print(f"\n{BLUE}Processing...{RESET}")
            response, messages = await query_claude(user_input, tools_local, messages)
            
            # Display Claude's response
            print(f"\n{BOLD}Assistant:{RESET} {response}")
            
    except Exception as e:
        print(f"\n{YELLOW}⚠️ An error occurred: {str(e)}{RESET}")

print("Chat session function defined. Run 'await chat_session()' in the next cell to start chatting.")

我们的 MCP host 界面：

1. **初始化工具**：我们的 host 在启动时发现可用工具
2. **创建会话循环**：持续提示用户输入
3. **维护上下文**：将之前的消息传递给每个查询，维护有状态的对话
4. **优雅退出处理**：允许用户优雅地结束会话

这创造了一种自然的对话体验，Agent 可以记住之前的交互。例如，如果用户询问了比特币，然后追问 "How about Ethereum?"，Agent 能够理解上下文。

现在，让我们运行聊天会话，看看完整的 Agent 的实际表现：

你可以试试我们在 Claude Desktop 中问过的问题：What's the market data for Dogecoin and Solana?

In [ ]:
# Run the chat session
await chat_session()

凭借这最后一块拼图，我们创建了一个完整的自定义 MCP host 实现，它可以：
1. 作为 client 连接到 MCP 服务器
2. 发现可用工具
3. 智能地选择并使用这些工具来回答用户查询
4. 在对话中维护上下文

这展示了实现我们自己的 MCP host 和 client 的威力——我们能够完全掌控 AI 与工具交互的方式，同时保留 MCP 协议标准化带来的所有好处。

## 结论：

Model Context Protocol 代表了一种将 AI 模型与外部资源集成的变革性方法，解决了 AI 应用开发中的关键挑战：

### 协议优势
- **标准化集成**：消除复杂、定制的 API 连接
- **动态工具发现**：让 AI 能够无缝发现并使用工具
- **灵活通信**：支持实时、双向交互

### 技术亮点
我们的实现演示了：
- 构建带专用工具的 MCP 服务器
- 创建能够动态发现和执行工具的 host
- 将 AI 模型与外部资源集成

### 关键引用

- [Python MCP SDK](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=github-python-sdk&target=https%3A%2F%2Fgithub.com%2Fmodelcontextprotocol%2Fpython-sdk&text=Python%20MCP%20SDK)
- [MCP Quick Start Guide](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=modelcontextprotocol-user&target=https%3A%2F%2Fmodelcontextprotocol.io%2Fquickstart%2Fuser&text=MCP%20Quick%20Start%20Guide)
- [CoinGecko API Documentation](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-mcp--mcp-tutorial&click=coingecko-api&target=https%3A%2F%2Fwww.coingecko.com%2Fen%2Fapi&text=CoinGecko%20API%20Documentation)